# Elaborative Rehearsal (B) — Training (stage 1)

Trains `t5-small` on the pairs from
`06_rehearsal_elaborative_prep.ipynb`. Structurally the same as
`05_rehearsal_maintenance_train.ipynb`; the manipulation check is inverted.

| | A (`05`) | B (this notebook) |
|---|---|---|
| target | sentences lifted from the input | a rewritten one-sentence summary |
| novel n-gram ratio should be | low, and only from seams | clearly **above** that |
| verbatim snapping after generation | yes (`_snap_to_verbatim`) | **no** — it would undo the elaboration |

**The manipulation check is the point of this notebook**, more than ROUGE-L.
The experiment claims two *different* rehearsal strategies; if B's output
turns out to be as copy-heavy as A's, then conditions A and B differ in name
only and every downstream comparison between them is empty. So novel n-gram
ratio is measured against the **source document**, not the label — comparing
against the label would be measuring how well B reproduces a summary that is
itself abstractive, which says nothing about whether B abstracts.

Prior work: RECOMP (Xu, Shi & Choi, ICLR 2024) §3.2 trains its abstractive
compressor as a summarization-pretrained seq2seq distilled from a larger
teacher; stage 1 here is the supervised half of that recipe, with XSum
standing in for the teacher's abstractive targets. The distillation and
selective-augmentation half arrives in stage 2.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(__import__("os").environ.get("NVIDIA_NIM_API_KEY"))

import datasets
import evaluate
import numpy as np
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

from src.pipeline.rehearsal import novel_ngram_ratio

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data/model

`val` drives checkpoint selection (`load_best_model_at_end` /
`metric_for_best_model="rougeL"`); `test` is not touched until §5.

In [2]:
MODEL_NAME = "t5-small"
DATA_DIR = Path("data/processed/rehearsal_elaborative")
OUTPUT_DIR = "experiments/rehearsal_elaborative_small"
TRAIN_SIZE = 3000
VAL_SIZE = 300
TEST_SIZE = 300

DATA_READY = all((DATA_DIR / split).exists() for split in ("train", "val", "test"))
if not DATA_READY:
    print(f"No tokenized data at {DATA_DIR} — run 06_rehearsal_elaborative_prep.ipynb first.")
else:
    full_train = datasets.Dataset.load_from_disk(str(DATA_DIR / "train"))
    full_val = datasets.Dataset.load_from_disk(str(DATA_DIR / "val"))
    full_test = datasets.Dataset.load_from_disk(str(DATA_DIR / "test"))

    train_dataset = full_train.select(range(min(TRAIN_SIZE, len(full_train))))
    val_dataset = full_val.select(range(min(VAL_SIZE, len(full_val))))
    test_dataset = full_test.select(range(min(TEST_SIZE, len(full_test))))
    print(f"Using {len(train_dataset)}/{len(full_train)} train, "
          f"{len(val_dataset)}/{len(full_val)} val, {len(test_dataset)}/{len(full_test)} test rows")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    print(f"{MODEL_NAME} loaded, parameters: {sum(p.numel() for p in model.parameters()):,}")

Using 3000/3000 train, 299/299 val, 300/300 test rows
t5-small loaded, parameters: 60,506,624


## 2. Metrics

`compute_metrics` reports ROUGE-L against the reference summary.

The novel n-gram ratio lives in a separate callback for the same reason as
`05`: it has to compare generations against the **source** `input_ids`, which
`compute_metrics`'s `(prediction, label)` pair does not provide. Here the
direction of the expectation is reversed — B should trend *away* from zero.

In [3]:
rouge = evaluate.load("rouge")


def compute_metrics(eval_preds):
    predictions, labels = eval_preds
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=False)
    return {"rougeL": result["rougeL"]}


def average_novel_ngram_ratio(model, tokenizer, eval_dataset, n: int = 3,
                              sample_size: int = 10, max_new_tokens: int = 64) -> float:
    """Generates from a slice of eval_dataset and returns the average novel
    n-gram ratio against the *source* (input), not the label. Shared by the
    callback and the final test evaluation so both use identical logic."""
    examples = eval_dataset.select(range(min(sample_size, len(eval_dataset))))
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device

    ratios = []
    for example in examples:
        input_ids = torch.tensor([example["input_ids"]]).to(device)
        with torch.no_grad():
            output_ids = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens)
        generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        source = tokenizer.decode(example["input_ids"], skip_special_tokens=True)
        ratios.append(novel_ngram_ratio(generated, source, n=n))

    if was_training:
        model.train()
    return sum(ratios) / len(ratios) if ratios else 0.0


class NovelNGramCallback(TrainerCallback):
    """Logs B's novel n-gram ratio on a validation slice at every eval.

    For B this should stay clearly above zero — it is the evidence that
    elaborative rehearsal is doing something A does not. A ratio drifting
    toward zero means B has collapsed into extraction and the A/B contrast
    has quietly disappeared.
    """

    def __init__(self, tokenizer, eval_dataset, n: int = 3, sample_size: int = 10, max_new_tokens: int = 64):
        self.tokenizer = tokenizer
        self.eval_dataset = eval_dataset
        self.n = n
        self.sample_size = sample_size
        self.max_new_tokens = max_new_tokens

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        ratio = average_novel_ngram_ratio(
            model, self.tokenizer, self.eval_dataset, n=self.n,
            sample_size=self.sample_size, max_new_tokens=self.max_new_tokens,
        )
        print(f"  [manipulation check] novel {self.n}-gram ratio vs source: {ratio:.4f} (higher is better for B)")

## 3. Train

`predict_with_generate=True` is required — ROUGE-L needs generated text, not
logits. `generation_max_length=64` matches `TARGET_MAX_LENGTH` from `06`
(XSum summaries are one sentence; `05` used 256 because A's targets are up to
three extracted sentences).

In [4]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=64,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[NovelNGramCallback(tokenizer, val_dataset)],
)

trainer.train()

  0%|          | 0/1125 [00:00<?, ?it/s]/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
  4%|▍         | 50/1125 [00:45<15:23,  1.16it/s]

{'loss': 3.4772, 'grad_norm': 2.7568604946136475, 'learning_rate': 4.7777777777777784e-05, 'epoch': 0.13}


  9%|▉         | 100/1125 [01:29<13:43,  1.25it/s]

{'loss': 3.2068, 'grad_norm': 2.6164329051971436, 'learning_rate': 4.555555555555556e-05, 'epoch': 0.27}


 13%|█▎        | 150/1125 [02:12<12:58,  1.25it/s]

...

{'eval_loss': 2.6944637298583984, 'eval_rougeL': 0.19127984727443306, 'eval_runtime': 70.95, 'eval_samples_per_second': 4.214, 'eval_steps_per_second': 0.536, 'epoch': 3.0}


  [manipulation check] novel 3-gram ratio vs source: 0.6673 (higher is better for B)


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].
100%|██████████| 1125/1125 [20:16<00:00,  1.08s/it]

{'train_runtime': 1216.0101, 'train_samples_per_second': 7.401, 'train_steps_per_second': 0.925, 'train_loss': 2.9964492730034724, 'epoch': 3.0}


TrainOutput(global_step=1125, training_loss=2.9964492730034724, metrics={'train_runtime': 1216.0101, 'train_samples_per_second': 7.401, 'train_steps_per_second': 0.925, 'total_flos': 735312007397376.0, 'train_loss': 2.9964492730034724, 'epoch': 3.0})

## 4. Save final model

In [5]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

Saved to: experiments/rehearsal_elaborative_small


## 5. Final evaluation — held-out test set

`test_dataset` was used for neither gradient updates nor checkpoint
selection. This is the number to report.

In [6]:
if not DATA_READY:
    print("No data — skipping final test evaluation.")
else:
    test_trainer = Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_eval_batch_size=8,
            predict_with_generate=True,
            generation_max_length=64,
            report_to="none",
        ),
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    test_metrics = test_trainer.evaluate(eval_dataset=test_dataset, metric_key_prefix="test")
    print("Final test-set metrics:", test_metrics)

    test_novel_ratio = average_novel_ngram_ratio(model, tokenizer, test_dataset, sample_size=30)
    print(f"Final test-set novel 3-gram ratio: {test_novel_ratio:.4f} (higher is better for B)")

/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 38/38 [01:02<00:00,  1.65s/it]


Final test-set metrics: {'test_loss': 2.7300515174865723, 'test_model_preparation_time': 0.0, 'test_rougeL': 0.1871442535080356, 'test_runtime': 67.5872, 'test_samples_per_second': 4.439, 'test_steps_per_second': 0.562}
Final test-set novel 3-gram ratio: 0.8003 (higher is better for B)


## 6. A/B contrast — the check that actually matters

Runs B and A over the **same chunks** from a held-out `cnn_dailymail` test
article (the same one `05` used, so the two notebooks' qualitative sections
are directly comparable) and puts their novel n-gram ratios side by side.

A single number for B in isolation is hard to interpret — 0.4 is only
meaningful next to A's 0.0. If the two come out close, the experiment does
not have two conditions, it has one condition run twice.

In [7]:
if not DATA_READY:
    print("No data — skipping.")
elif not API_KEY_SET:
    print("No NVIDIA_NIM_API_KEY (chunking needs the embedding API) — skipping the A/B contrast.")
else:
    import yaml
    from datasets import load_dataset

    from src.pipeline.chuncking import paginate_semantic, plain_text_to_paragraphs
    from src.pipeline.embeddings import embed_texts, load_config

    test_article = load_dataset("cnn_dailymail", "3.0.0", split="test[0:1]")[0]["article"]
    chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
    embed_cfg = load_config("configs/importance_filter.yaml")
    chunks = paginate_semantic(
        plain_text_to_paragraphs(test_article),
        min_words=chunk_cfg["min_words"], max_words=chunk_cfg["max_words"],
        granularity="paragraph", config=embed_cfg, embed_fn=embed_texts,
    )[:5]
    print(f"chunks: {len(chunks)}\n")

    device = next(model.parameters()).device
    rows = []
    for chunk in chunks:
        ids = tokenizer(chunk.text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=64)
        elaborated = tokenizer.decode(out[0], skip_special_tokens=True)
        rows.append(
            {
                "chunk": chunk.index,
                "source_chars": len(chunk.text),
                "output_chars": len(elaborated),
                "compression": round(len(elaborated) / max(1, len(chunk.text)), 3),
                "novel_3gram": round(novel_ngram_ratio(elaborated, chunk.text, n=3), 4),
                "text": elaborated[:110],
            }
        )

    import pandas as pd

    contrast = pd.DataFrame(rows)
    print(contrast.to_string(index=False))
    print(f"\nB mean novel 3-gram ratio: {contrast.novel_3gram.mean():.4f}")
    print("A (05, same article) stays near zero and only from sentence seams.")
    print("A clear gap here is the manipulation check passing.")

chunks: 5

 chunk  source_chars  output_chars  compression  novel_3gram                                                                                                           text
     0           717           120        0.167       0.6000 The Palestinian Authority has officially become the 123rd member of the International Criminal Court in the Ne
     1           593           104        0.175       0.8571       Palestinians are being re-elected as members of the Rome Court, a Palestinian Foreign Minister has said.
     2           726            90        0.124       1.0000                     Palestinians have a "substantial commitment" to the Rome Statute, a rights group has said.
     3           619            90        0.145       0.6154                     The United States has said it "strongly" disagreed with the decision of the Israeli court.
     4           953           314        0.329       0.9167 The International Criminal Court has ruled that the Palestinian terr

## Summary

_To be filled in after the run._

What to look at, in order:

1. **Novel 3-gram ratio vs. source** (§5, §6). B must sit clearly above A's
   0.0000. This is the manipulation check; without it the A/B distinction is
   not established and conditions A, B, A+C, B+C stop being four distinct
   things.
2. **Compression ratio** (§6). XSum targets are one sentence, so B compresses
   far harder than A did (A ran 22-32% of source characters). Very aggressive
   compression is expected here — the open question is whether it keeps
   enough to answer questions later, which only the pipeline run can settle.
3. **test ROUGE-L** (§5). Report this, not the validation number.

Known limitation, by design: stage 1 teaches abstraction on single documents
and cannot teach cross-chunk integration, because `(document → summary)`
pairs have no preceding chunk. Stage 2 — rolling `(S_{i-1}, C_i) → S_i`
curation with teacher distillation, self-generated probes, a corrective
retry, and selective augmentation against a verbatim fallback — is specified
in `03_실험 설계` §정교화 되뇌기(B) 구현 설계 and needs the QG model from
`09` first.